NOTE: This file runs the e010_pjt_usgs_flowlines main Prefect flow, but it passes it the network_type=all undirected OSM map.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path

In [3]:
from common.osm.enrich import create_full_enriched_osmnx_graph_for_region, convert_graph_to_gdfs

In [4]:
from experiments.e010_pjt_usgs_flowlines.tasks.complete_prefect_workflow import HydrologicalHazardStrategy, roadway_hydrological_hazard_flow

/home/avail/AVAIL/avail-research-road-network-resiliency/.venv/lib/python3.12/site-packages/pydantic/json_schema.py:2324: PydanticJsonSchemaWarning: Default value HydrologicalHazardStrategy(enrich_osm=<prefect.tasks.Task object at 0x77496e34b9e0>, create_combined_road_spans=<prefect.tasks.Task object at 0x77496e3f8140>, get_clipped_nhdplus_flowlines=<prefect.tasks.Task object at 0x77496e3f8380>, process_road_flowline_intersections=<prefect.tasks.Task object at 0x77496e3cbe00>, assign_simple_hydrological_risk=<prefect.tasks.Task object at 0x7749701c8740>, aggregate_risks=<prefect.tasks.Task object at 0x77496e348a40>, calculate_composite_roadway_hazard_score=<prefect.tasks.Task object at 0x7749c8ebfe90>, prepare_roads_for_composite_merge=<prefect.tasks.Task object at 0x77496e37d1f0>, merge_composite_hazard_score=<prefect.tasks.Task object at 0x77496e3c8680>, save_outputs=<prefect.tasks.Task object at 0x77496e3d55e0>) is not JSON serializable; excluding default from JSON schema [non-seria

In [5]:
# osm_pbf = '../data/processed/all-nonparking-highways-county-36001_new-york-240101.osm.pbf'
osm_pbf = Path(
    "../../../data/processed/osm/" 
    "all-ways-buffer-10mi-county-36001_"
    "buffer-50mi-state-36_all-nonparking-highways-us-250101.osm.pbf"
)

nhd_flowlines_path = Path(
    "../../../data/raw/usgs/national_hydrology_dataset/NHDPlus_H_National_Release_2_GDB.zip"
)

In [6]:
enriched_osm = create_full_enriched_osmnx_graph_for_region(
    osm_pbf=osm_pbf,
    network_type='all'
)

In [7]:
G = enriched_osm["G"]
g = enriched_osm["g"]

In [8]:
# undirected_G = G.to_undirected()
undirected_G = G

In [9]:
undirected_g = g.to_undirected()

In [10]:
undirected_nodes_gdf, undirected_edges_gdf = convert_graph_to_gdfs(undirected_g) # type: ignore

In [11]:
undirected_enriched_osm = enriched_osm | {
    "G": undirected_G,
    "g": undirected_g,
    "nodes_gdf": undirected_nodes_gdf,
    "edges_gdf": undirected_edges_gdf,
}

In [12]:
undirected_edges_gdf["_intersects_region_"].value_counts()

_intersects_region_
False    88268
True     61259
Name: count, dtype: int64

In [13]:
undirected_edges_gdf["geometry"]

u            v            key
29746265     443146532    0      LINESTRING (-74.01188 42.82814, -74.01212 42.8...
             1686423655   0      LINESTRING (-74.01832 42.80395, -74.01912 42.8...
29746351     1686423661   0      LINESTRING (-74.01806 42.80398, -74.01769 42.8...
41247986     41454684     0      LINESTRING (-73.94586 42.61767, -73.94614 42.6...
             8110409390   0      LINESTRING (-73.94955 42.6166, -73.94952 42.61...
                                                       ...                        
12465993036  12465993034  0      LINESTRING (-73.75652 42.66389, -73.75608 42.6...
12466212225  12466212238  0      LINESTRING (-73.82224 42.37772, -73.82237 42.3...
12466212238  12466212256  0      LINESTRING (-73.81994 42.37412, -73.81979 42.3...
             12466253573  0      LINESTRING (-73.82154 42.37441, -73.82157 42.3...
12466212256  12466253573  0      LINESTRING (-73.82154 42.37441, -73.82146 42.3...
Name: geometry, Length: 149527, dtype: geometry

In [14]:
def get_undirected_enriched_osm(**kwargs):
    return undirected_enriched_osm

In [15]:
strategy = HydrologicalHazardStrategy(enrich_osm=get_undirected_enriched_osm)

In [16]:
roadway_hydrological_hazard_flow(
    osm_pbf=Path("foo"),
    nhd_flowlines_path=nhd_flowlines_path,
    clean=True,
    output_gpkg=Path("undirected_all_nonparking_roadways_hydrological_hazard_flow.gpkg"),
    strategy=strategy
)

16:54:36.285 | INFO    | prefect - Starting temporary server on http://127.0.0.1:8044
See https://docs.prefect.io/3.0/manage/self-host#self-host-a-prefect-server for more information on running a dedicated Prefect server.

16:54:40.877 | INFO    | Flow run 'faithful-wapiti' - Beginning flow run 'faithful-wapiti' for flow 'Roadway Hydrological Hazard Workflow'

2025-06-28 16:54:40 - INFO - prefect.flow_runs - --- Starting Roadway Hydrological Hazard Workflow ---
2025-06-28 16:54:40 - INFO - prefect.flow_runs - OSM Input: foo
2025-06-28 16:54:40 - INFO - prefect.flow_runs - NHD Flowlines Input: ../../../data/raw/usgs/national_hydrology_dataset/NHDPlus_H_National_Release_2_GDB.zip
2025-06-28 16:54:40 - INFO - prefect.flow_runs - Clean Run: True
2025-06-28 16:54:40 - INFO - prefect.flow_runs - Verbose Logging: False
2025-06-28 16:54:42 - INFO - httpx - HTTP Request: GET http://127.0.0.1:8044/api/flows/28d9669e-5039-4eaf-943e-832a12edcfd7 "HTTP/1.1 200 OK"
2025-06-28 16:54:42 - INFO - prefect.task_runs - Combining bridge and non-bridge spans.
2025-06-28 16:54:42 - INFO - httpx - HTTP Request: GET http://127.0.0.1:8044/api/csrf-token?client=2019e904-6c23-45df-92fc-96d4477e77f3 "HTTP/1.1 422 Unprocessable Entity"
2025-06-28 16:54:43 - INFO - httpx - HTTP Request: POST http://127.0.0.1:8044/api/logs/ "HTTP/1.1 201 Created"
2025-06-28 16:55:11 - INFO

'undirected_all_nonparking_roadways_hydrological_hazard_flow.gpkg'

2025-06-28 16:56:33 - INFO - httpx - HTTP Request: POST http://127.0.0.1:8044/api/logs/ "HTTP/1.1 201 Created"
